In [ ]:
import dt4dds_benchmark
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

data = dt4dds_benchmark.analysis.Dataset.combine(*[dt4dds_benchmark.pipelines.HDF5Manager(f'./data/{s}.hdf5').get_data() for s in (
    'aeon_high',
    'aeon_low',
    'aeon_medium',
    'fountain_high',
    'fountain_low',
    'fountain_medium',
    'goldman_default',
    'rs_high',
    'rs_low',
    'rs_medium',
    'hedges_low',
    'hedges_medium',
    'yinyang_default',
)])

# generate metrics

In [ ]:
df = data.combined_results.copy()
df['codecname'] = df['codec.type'] + df['codec.name']
df['p_recovered'] = df['total_foundreferences'] / df['n_sequences']
df['p_clusterperreference'] = df['total_clusters'] / df['n_sequences']
df['p_min_errors'] = df['min_editdistance'] / df['sequence_length']
df['p_errors'] = df['mean_editdistance'] / df['sequence_length']

df = df.sort_values(by=['codec.type', 'codec.name', 'clustering.type', 'clustering.name', 'workflow.overall_rate'])

### export raw data for effective error rates for the main figure

In [ ]:
df[['codec.type', 'codec.name', 'clustering.type', 'workflow.overall_rate', 'p_min_errors']].to_csv('error_rates.csv', index=False)

In [ ]:
# save data
df[['codec.type', 'codec.name', 'clustering.type', 'workflow.overall_rate', 'p_min_errors', 'p_errors', 'p_clusterperreference']].to_csv('./figures/analysis.csv', index=False)

### fix ordering

In [ ]:
df['codec.name'] = df['codec.name'].map({
    'default': '1low',
    'low': '1low',
    'medium': '2medium',
    'high': '3high',
})
df.loc[df['codec.type'] == 'YinYang', 'codec.name'] = '3high'
df = df.sort_values(by=['codec.type', 'codec.name', 'clustering.type', 'clustering.name'])

# plot workflow rate vs. overall measured rate

In [ ]:
fig = px.line(
    df,
    x='workflow.overall_rate',
    y='p_errors',
    facet_col='codec.name',
    facet_col_spacing=0.05,
    facet_row='codec.type',
    facet_row_spacing=0.03,
    color='clustering.type',
)
fig.update_xaxes(range=[0, 0.15])
fig.update_yaxes(range=[0, 0.15])
fig.update_layout(
    width=680,
    height=800,
    margin=dict(l=0, r=10, t=20, b=10),
    showlegend=False,
)
fig.update_xaxes(title='Applied error rate', row=1)
fig.update_xaxes(title='Applied error rate', row=2)
fig.update_xaxes(showticklabels=True, row=2)
fig.update_yaxes(title='Effective error rate', col=1)
fig.update_yaxes(showticklabels=True, row=1, col=3)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig = dt4dds_benchmark.analysis.plotting.standardize_plot(fig)

fig.write_image('./figures/rate_per_sequence.svg')
fig.show()

# plot workflow rate vs. minimum effective rate

In [ ]:
fig = px.line(
    df,
    x='workflow.overall_rate',
    y='p_min_errors',
    facet_col='codec.name',
    facet_col_spacing=0.05,
    facet_row='codec.type',
    facet_row_spacing=0.03,
    color='clustering.type',
)
fig.update_xaxes(range=[0, 0.15])
fig.update_yaxes(range=[0, 0.15])
fig.update_layout(
    width=680,
    height=800,
    margin=dict(l=0, r=10, t=20, b=10),
    showlegend=False,
)
fig.update_xaxes(title='Applied error rate', row=1)
fig.update_xaxes(title='Applied error rate', row=2)
fig.update_xaxes(showticklabels=True, row=2)
fig.update_yaxes(title='Effective error rate', col=1)
fig.update_yaxes(showticklabels=True, row=1, col=3)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig = dt4dds_benchmark.analysis.plotting.standardize_plot(fig)

fig.write_image('./figures/min_rate_per_sequence.svg')
fig.show()

# plot workflow rate vs. cluster size

In [ ]:
fig = px.line(
    df,
    x='workflow.overall_rate',
    y='p_clusterperreference',
    facet_col='codec.name',
    facet_col_spacing=0.05,
    facet_row='codec.type',
    facet_row_spacing=0.03,
    color='clustering.type',
)
fig.update_xaxes(range=[0, 0.15])
fig.update_yaxes(range=[0, 31])
fig.update_layout(
    width=680,
    height=800,
    margin=dict(l=0, r=10, t=20, b=10),
    showlegend=False,
)
fig.update_xaxes(title='Applied error rate', row=1)
fig.update_xaxes(title='Applied error rate', row=2)
fig.update_xaxes(showticklabels=True, row=2)
fig.update_yaxes(title='Clusters per sequence', col=1)
fig.update_yaxes(showticklabels=True, row=1, col=3)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig = dt4dds_benchmark.analysis.plotting.standardize_plot(fig)

fig.write_image('./figures/clusters_per_sequence.svg')
fig.show()

# tabulate the ratio of missing sequences

In [ ]:
tabledf = pd.pivot_table(
    df.loc[df['workflow.overall_rate'] == 0.0],
    index=['codec.type', 'codec.name'],
    columns='clustering.type',
    values='p_recovered',
)

# format to two decimal places
tabledf = tabledf.map(lambda x: f"{100*x:.1f}" if isinstance(x, (float, np.float64)) else x)

tabledf.to_csv('./figures/recovery_rates.csv', index=True)
tabledf